In [0]:
import os
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from mlflow import MlflowClient

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
EXPERIMENT_NAME = "/Shared/Football_MLflow_Experiment"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())
print("Experiment:", EXPERIMENT_NAME)

Tracking URI: databricks
Registry URI: databricks-uc
Experiment: /Shared/Football_MLflow_Experiment


In [0]:
DATA_DIRECTORY = "/Volumes/workspace/default/football_data"

TEST_PATH = (
    f"{DATA_DIRECTORY}/football_test.csv"
)

CHAMPION_INFO_PATH = (
    f"{DATA_DIRECTORY}/champion_run_info.csv"
)

CHAMPION_REGISTRATION_PATH = (
    f"{DATA_DIRECTORY}/champion_registration_info.csv"
)

CHALLENGER_INFO_PATH = (
    f"{DATA_DIRECTORY}/challenger_run_info.csv"
)

CHALLENGER_SUMMARY_PATH = (
    f"{DATA_DIRECTORY}/challenger_evaluation_summary.csv"
)

PROMOTION_RECORD_PATH = (
    f"{DATA_DIRECTORY}/promotion_record.csv"
)

MODEL_HISTORY_PATH = (
    f"{DATA_DIRECTORY}/model_history.csv"
)

NEW_CHAMPION_INFO_PATH = (
    f"{DATA_DIRECTORY}/new_champion_info.csv"
)

REGISTERED_MODEL_NAME = (
    "workspace.default.football_match_result_model"
)

CHAMPION_ALIAS = "champion"

TARGET_COLUMN = "match_result"

LEAKAGE_COLUMNS = [
    "home_score",
    "away_score"
]

PROMOTION_METRIC = "test_f1_macro"

MINIMUM_REQUIRED_IMPROVEMENT = 0.0

print("Registered model:", REGISTERED_MODEL_NAME)
print("Promotion metric:", PROMOTION_METRIC)
print("Minimum required improvement:", MINIMUM_REQUIRED_IMPROVEMENT)

Registered model: workspace.default.football_match_result_model
Promotion metric: test_f1_macro
Minimum required improvement: 0.0


In [0]:
required_files = [
    TEST_PATH,
    CHAMPION_INFO_PATH,
    CHAMPION_REGISTRATION_PATH,
    CHALLENGER_INFO_PATH,
    CHALLENGER_SUMMARY_PATH
]

missing_files = [
    file_path
    for file_path in required_files
    if not os.path.exists(file_path)
]

assert not missing_files, (
    "The following required files are missing: "
    f"{missing_files}"
)

print("All required files are available.")

All required files are available.


In [0]:
champion_info_df = pd.read_csv(
    CHAMPION_INFO_PATH
)

champion_registration_df = pd.read_csv(
    CHAMPION_REGISTRATION_PATH
)

challenger_info_df = pd.read_csv(
    CHALLENGER_INFO_PATH
)

challenger_summary_df = pd.read_csv(
    CHALLENGER_SUMMARY_PATH
)

print("Current Champion:")
display(champion_info_df)

print("Challenger:")
display(challenger_info_df)

print("Notebook 6 comparison:")
display(challenger_summary_df)

Current Champion:


model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,max_depth,min_samples_split,min_samples_leaf,class_weight,removed_columns_path,registration_status,alias_status,registered_model_name,registered_model_version,registered_model_alias,alias_model_uri
champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,models:/m-a7c158f4672f45faad9513d569e68b0c,test_f1_macro,0.3407115240453506,0.35575,0.5227480171428843,0.4249912752465563,0.3407115240453506,0.3219326636027983,15999,4000,0.334583044052124,3,20,10,balanced,/Volumes/workspace/default/football_data/champion_removed_columns.json,READY,assigned,workspace.default.football_match_result_model,4,champion,models:/workspace.default.football_match_result_model@champion


Challenger:


model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,prediction_time_seconds,champion_algorithm,champion_run_id,champion_model_version,champion_macro_f1,macro_f1_improvement,relative_improvement_percentage,promotion_eligible,promotion_recommendation,registration_status,registered_model_version,alias_status
challenger,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,test_f1_macro,0.6222097633743984,0.6515,0.6446662708271728,0.6136885178601231,0.6222097633743984,0.6434775072688667,15999,4000,25.322230577468872,0.0281131267547607,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506,0.2814982393290478,82.62069799892615,true,PROMOTE_CHALLENGER,pending,null,not_assigned


Notebook 6 comparison:


comparison_metric,champion_algorithm,champion_macro_f1,challenger_algorithm,challenger_macro_f1,absolute_improvement,relative_improvement_percentage,challenger_outperforms_champion,promotion_eligible,recommendation
Macro F1,DecisionTreeClassifier,0.3407115240453506,LogisticRegression,0.6222097633743984,0.2814982393290478,82.62069799892615,true,true,PROMOTE_CHALLENGER


In [0]:
assert len(champion_info_df) == 1, (
    "Champion information must contain exactly one row."
)

assert len(champion_registration_df) == 1, (
    "Champion registration information must contain exactly one row."
)

assert len(challenger_info_df) == 1, (
    "Challenger information must contain exactly one row."
)

assert len(challenger_summary_df) == 1, (
    "Challenger summary must contain exactly one row."
)

print("Metadata row counts validated.")

Metadata row counts validated.


In [0]:
required_champion_columns = [
    "algorithm",
    "run_id",
    "model_uri",
    "primary_metric",
    "primary_metric_value",
    "registered_model_version"
]

required_challenger_columns = [
    "algorithm",
    "run_id",
    "model_uri",
    "primary_metric",
    "primary_metric_value",
    "champion_macro_f1",
    "macro_f1_improvement",
    "promotion_eligible",
    "promotion_recommendation"
]

missing_champion_columns = [
    column
    for column in required_champion_columns
    if column not in champion_info_df.columns
]

missing_challenger_columns = [
    column
    for column in required_challenger_columns
    if column not in challenger_info_df.columns
]

assert not missing_champion_columns, (
    "Missing Champion columns: "
    f"{missing_champion_columns}"
)

assert not missing_challenger_columns, (
    "Missing Challenger columns: "
    f"{missing_challenger_columns}"
)

print("Required metadata columns validated.")

Required metadata columns validated.


In [0]:
champion_record = champion_info_df.iloc[0]

old_champion_algorithm = str(
    champion_record["algorithm"]
)

old_champion_run_id = str(
    champion_record["run_id"]
)

old_champion_model_uri = str(
    champion_record["model_uri"]
)

old_champion_version = str(
    champion_record["registered_model_version"]
)

old_champion_metric_name = str(
    champion_record["primary_metric"]
)

old_champion_macro_f1 = float(
    champion_record["primary_metric_value"]
)

print("Current Champion algorithm:", old_champion_algorithm)
print("Current Champion version:", old_champion_version)
print("Current Champion run ID:", old_champion_run_id)
print("Current Champion metric:", old_champion_metric_name)

print(
    "Current Champion Macro F1:",
    round(old_champion_macro_f1, 4)
)
assert old_champion_algorithm == "DecisionTreeClassifier"

assert old_champion_metric_name == PROMOTION_METRIC

print("Current Champion metadata validated.")

Current Champion algorithm: DecisionTreeClassifier
Current Champion version: 4
Current Champion run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
Current Champion metric: test_f1_macro
Current Champion Macro F1: 0.3407
Current Champion metadata validated.


In [0]:
challenger_record = challenger_info_df.iloc[0]

challenger_algorithm = str(
    challenger_record["algorithm"]
)

challenger_run_id = str(
    challenger_record["run_id"]
)

challenger_model_uri = str(
    challenger_record["model_uri"]
)

challenger_metric_name = str(
    challenger_record["primary_metric"]
)

challenger_macro_f1 = float(
    challenger_record["primary_metric_value"]
)

notebook6_improvement = float(
    challenger_record["macro_f1_improvement"]
)

saved_promotion_eligible = str(
    challenger_record["promotion_eligible"]
).strip().lower() == "true"

saved_promotion_recommendation = str(
    challenger_record["promotion_recommendation"]
)

print("Challenger algorithm:", challenger_algorithm)
print("Challenger run ID:", challenger_run_id)
print("Challenger model URI:", challenger_model_uri)
print("Challenger metric:", challenger_metric_name)

print(
    "Challenger Macro F1:",
    round(challenger_macro_f1, 4)
)
assert challenger_algorithm == "LogisticRegression"

assert challenger_metric_name == PROMOTION_METRIC

assert saved_promotion_eligible

assert (
    saved_promotion_recommendation
    == "PROMOTE_CHALLENGER"
)

print("Challenger metadata validated.")

Challenger algorithm: LogisticRegression
Challenger run ID: 80ac2ed817e940c9a66f4e99f15b08f4
Challenger model URI: models:/m-d7e371663ec74ae8930885ebaa2a6cc2
Challenger metric: test_f1_macro
Challenger Macro F1: 0.6222
Challenger metadata validated.


In [0]:
recalculated_improvement = (
    challenger_macro_f1
    - old_champion_macro_f1
)

relative_improvement_percentage = (
    recalculated_improvement
    / old_champion_macro_f1
    * 100
)

promotion_eligible = (
    challenger_macro_f1
    > (
        old_champion_macro_f1
        + MINIMUM_REQUIRED_IMPROVEMENT
    )
)

print("=" * 65)
print("PROMOTION ELIGIBILITY RECHECK")
print("=" * 65)

print(
    "Champion Macro F1:",
    round(old_champion_macro_f1, 4)
)

print(
    "Challenger Macro F1:",
    round(challenger_macro_f1, 4)
)

print(
    "Absolute improvement:",
    round(recalculated_improvement, 4)
)

print(
    "Relative improvement:",
    round(relative_improvement_percentage, 2),
    "%"
)

print(
    "Promotion eligible:",
    promotion_eligible
)

assert np.isclose(
    recalculated_improvement,
    notebook6_improvement,
    atol=1e-8
), (
    "The recalculated improvement does not match "
    "the Notebook 6 result."
)

assert promotion_eligible, (
    "The Challenger is not eligible for promotion. "
    "The Champion alias will not be changed."
)

print("Promotion eligibility independently confirmed.")

PROMOTION ELIGIBILITY RECHECK
Champion Macro F1: 0.3407
Challenger Macro F1: 0.6222
Absolute improvement: 0.2815
Relative improvement: 82.62 %
Promotion eligible: True
Promotion eligibility independently confirmed.


In [0]:
client = MlflowClient()

print("MLflow client created successfully.")

MLflow client created successfully.


In [0]:
pre_promotion_alias_model = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS
    )
)

pre_promotion_alias_version = str(
    pre_promotion_alias_model.version
)

pre_promotion_alias_run_id = str(
    pre_promotion_alias_model.run_id
)

print("Alias before promotion:", CHAMPION_ALIAS)
print("Version before promotion:", pre_promotion_alias_version)
print("Run ID before promotion:", pre_promotion_alias_run_id)

Alias before promotion: champion
Version before promotion: 4
Run ID before promotion: 3fa9c04ecf3d4f1597aa3bd68f650d2f


In [0]:
assert (
    pre_promotion_alias_version
    == old_champion_version
), (
    "The current champion alias version does not match "
    "the saved Champion metadata."
)

assert (
    pre_promotion_alias_run_id
    == old_champion_run_id
), (
    "The current champion alias run ID does not match "
    "the saved Champion metadata."
)

print(
    "Before promotion, the champion alias correctly "
    "points to the Decision Tree."
)

Before promotion, the champion alias correctly points to the Decision Tree.


In [0]:
logged_challenger_model = (
    mlflow.sklearn.load_model(
        challenger_model_uri
    )
)

print("Logged Challenger loaded successfully.")
print("Loaded object type:", type(logged_challenger_model))

Logged Challenger loaded successfully.
Loaded object type: <class 'sklearn.pipeline.Pipeline'>


In [0]:
if hasattr(logged_challenger_model, "named_steps"):

    loaded_classifier = (
        logged_challenger_model.named_steps[
            "classifier"
        ]
    )

    loaded_classifier_name = (
        type(loaded_classifier).__name__
    )

    print(
        "Loaded Challenger classifier:",
        loaded_classifier_name
    )

    assert (
        loaded_classifier_name
        == "LogisticRegression"
    )

print("Logged Challenger model verified.")

Loaded Challenger classifier: LogisticRegression
Logged Challenger model verified.


In [0]:
test_df = pd.read_csv(
    TEST_PATH
)

assert TARGET_COLUMN in test_df.columns

X_test = test_df.drop(
    columns=[
        TARGET_COLUMN,
        *[
            column
            for column in LEAKAGE_COLUMNS
            if column in test_df.columns
        ]
    ],
    errors="ignore"
).copy()

print("Test feature shape:", X_test.shape)

Test feature shape: (4000, 57)


In [0]:
challenger_model_info = mlflow.models.get_model_info(
    challenger_model_uri
)

challenger_signature = (
    challenger_model_info.signature
)

print("Challenger signature:")
print(challenger_signature)

Challenger signature:
inputs: 
  ['Country': string (required), 'League': string (required), 'home_team': string (required), 'away_team': string (required), 'expected_goals_xg_home': double (optional), 'expected_goals_xg_host': double (optional), 'Ball_Possession_Home': double (optional), 'Ball_Possession_Host': double (optional), 'Goal_Attempts_Home': double (optional), 'Goal_Attempts_Host': double (optional), 'Shots_on_Goal_Home': double (optional), 'Shots_on_Goal_Host': double (optional), 'Shots_off_Goal_Home': double (optional), 'Shots_off_Goal_Host': double (optional), 'Blocked_Shots_Home': double (optional), 'Blocked_Shots_Host': double (optional), 'Free_Kicks_Home': double (optional), 'Free_Kicks_Host': double (optional), 'Corner_Kicks_Home': double (optional), 'Corner_Kicks_Host': double (optional), 'Offsides_Home': double (optional), 'Offsides_Host': double (optional), 'Throw_ins_Home': double (optional), 'Throw_ins_Host': double (optional), 'Goalkeeper_Saves_Home': double (op

In [0]:
if (
    challenger_signature is not None
    and challenger_signature.inputs is not None
):

    signature_input_names = [
        input_spec.name
        for input_spec in challenger_signature.inputs.inputs
        if input_spec.name is not None
    ]

    missing_signature_columns = [
        column
        for column in signature_input_names
        if column not in X_test.columns
    ]

    assert not missing_signature_columns, (
        "Test data is missing model-signature columns: "
        f"{missing_signature_columns}"
    )

    X_test_aligned = X_test[
        signature_input_names
    ].copy()

else:
    X_test_aligned = X_test.copy()

print(
    "Aligned test feature shape:",
    X_test_aligned.shape
)

Aligned test feature shape: (4000, 44)


In [0]:
verification_sample = (
    X_test_aligned.head(20).copy()
)

pre_registration_predictions = (
    logged_challenger_model.predict(
        verification_sample
    )
)

print("Pre-registration predictions:")
print(pre_registration_predictions)

assert len(pre_registration_predictions) == 20

print("Pre-registration inference passed.")

Pre-registration predictions:
['Draw' 'Draw' 'Away Win' 'Home Win' 'Home Win' 'Home Win' 'Away Win'
 'Home Win' 'Home Win' 'Home Win' 'Home Win' 'Away Win' 'Away Win'
 'Home Win' 'Away Win' 'Draw' 'Home Win' 'Away Win' 'Home Win' 'Home Win']
Pre-registration inference passed.


In [0]:
existing_challenger_version = None

try:
    all_model_versions = (
        client.search_model_versions(
            f"name='{REGISTERED_MODEL_NAME}'"
        )
    )

    matching_versions = [
        model_version
        for model_version in all_model_versions
        if str(model_version.run_id)
        == challenger_run_id
    ]

    if matching_versions:

        existing_challenger_version = str(
            max(
                matching_versions,
                key=lambda version: int(
                    version.version
                )
            ).version
        )

        print(
            "This Challenger run is already registered."
        )

        print(
            "Existing Challenger version:",
            existing_challenger_version
        )

    else:
        print(
            "The Challenger run has not been "
            "registered previously."
        )

except Exception as error:
    print(
        "Could not complete duplicate registration check."
    )

    print("Details:", str(error))

The Challenger run has not been registered previously.


In [0]:
if existing_challenger_version is None:

    print(
        "Registering the Logistic Regression "
        "Challenger..."
    )

    registration_result = mlflow.register_model(
        model_uri=challenger_model_uri,
        name=REGISTERED_MODEL_NAME
    )

    new_champion_version = str(
        registration_result.version
    )

    print("Registration request completed.")
    print("New model version:", new_champion_version)
    print("Initial status:", registration_result.status)

else:

    new_champion_version = (
        existing_challenger_version
    )

    print(
        "Reusing the previously registered "
        "Challenger version."
    )

    print(
        "Challenger version:",
        new_champion_version
    )

Registering the Logistic Regression Challenger...


Registered model 'workspace.default.football_match_result_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '5' of model 'workspace.default.football_match_result_model': https://dbc-f41d3394-c5eb.cloud.databricks.com/explore/data/models/workspace/default/football_match_result_model/version/5?o=7474658572124013


Registration request completed.
New model version: 5
Initial status: READY


In [0]:
MAX_WAIT_SECONDS = 180
POLL_INTERVAL_SECONDS = 5

waited_seconds = 0

while waited_seconds <= MAX_WAIT_SECONDS:

    new_version_details = (
        client.get_model_version(
            name=REGISTERED_MODEL_NAME,
            version=new_champion_version
        )
    )

    current_registration_status = str(
        new_version_details.status
    )

    print(
        "Registration status:",
        current_registration_status
    )

    if current_registration_status == "READY":
        print("New model version is ready.")
        break

    if (
        current_registration_status
        == "FAILED_REGISTRATION"
    ):
        raise RuntimeError(
            "Challenger model registration failed. "
            f"Status message: "
            f"{new_version_details.status_message}"
        )

    time.sleep(POLL_INTERVAL_SECONDS)

    waited_seconds += POLL_INTERVAL_SECONDS

else:
    raise TimeoutError(
        "The Challenger model version did not become "
        f"READY within {MAX_WAIT_SECONDS} seconds."
    )

Registration status: READY
New model version is ready.


In [0]:
new_version_details = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=new_champion_version
    )
)

registered_challenger_run_id = str(
    new_version_details.run_id
)

print("Registered model:", new_version_details.name)
print("New version:", new_version_details.version)
print("Status:", new_version_details.status)
print("Run ID:", registered_challenger_run_id)

Registered model: workspace.default.football_match_result_model
New version: 5
Status: READY
Run ID: 80ac2ed817e940c9a66f4e99f15b08f4


In [0]:
assert (
    registered_challenger_run_id
    == challenger_run_id
), (
    "The registered version does not belong to "
    "the expected Challenger run."
)

assert str(
    new_version_details.status
) == "READY"

assert (
    new_champion_version
    != old_champion_version
), (
    "The new Challenger version must differ from "
    "the old Champion version."
)

print("Registered Challenger version validated.")

Registered Challenger version validated.


In [0]:
challenger_version_description = (
    "Logistic Regression Challenger promoted after "
    f"achieving Macro F1={challenger_macro_f1:.4f}, "
    f"compared with the previous Decision Tree "
    f"Champion Macro F1={old_champion_macro_f1:.4f}. "
    f"Absolute improvement={recalculated_improvement:.4f}."
)

client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=new_champion_version,
    description=challenger_version_description
)

print("Challenger model-version description updated.")

Challenger model-version description updated.


In [0]:
promotion_timestamp_utc = (
    datetime.now(timezone.utc)
    .isoformat()
)

new_champion_tags = {
    "model_role": "champion",
    "algorithm": "LogisticRegression",
    "workflow": "champion_challenger",
    "workflow_version": "version_2",
    "model_stage": "promoted_champion",
    "primary_metric": PROMOTION_METRIC,
    "primary_metric_value": challenger_macro_f1,
    "previous_champion_algorithm": old_champion_algorithm,
    "previous_champion_version": old_champion_version,
    "previous_champion_run_id": old_champion_run_id,
    "previous_champion_macro_f1": old_champion_macro_f1,
    "macro_f1_improvement": recalculated_improvement,
    "relative_improvement_percentage": (
        relative_improvement_percentage
    ),
    "promotion_decision": "PROMOTED",
    "promotion_notebook": (
        "07_Automatic_Challenger_Promotion"
    ),
    "promotion_timestamp_utc": (
        promotion_timestamp_utc
    )
}

for tag_key, tag_value in (
    new_champion_tags.items()
):

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=new_champion_version,
        key=tag_key,
        value=str(tag_value)
    )

print("New Champion tags added.")

New Champion tags added.


In [0]:
old_champion_tags = {
    "model_role": "previous_champion",
    "previous_alias_status": "replaced",
    "replacement_algorithm": (
        challenger_algorithm
    ),
    "replaced_by_version": (
        new_champion_version
    ),
    "replacement_run_id": (
        challenger_run_id
    ),
    "replacement_macro_f1": (
        challenger_macro_f1
    ),
    "retained_for": (
        "lineage_and_rollback"
    ),
    "promotion_timestamp_utc": (
        promotion_timestamp_utc
    )
}

for tag_key, tag_value in (
    old_champion_tags.items()
):

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=old_champion_version,
        key=tag_key,
        value=str(tag_value)
    )

print("Previous Champion tags updated.")

Previous Champion tags updated.


In [0]:
NEW_VERSION_MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"/{new_champion_version}"
)

print(
    "New version model URI:",
    NEW_VERSION_MODEL_URI
)

New version model URI: models:/workspace.default.football_match_result_model/5


In [0]:
registered_challenger_model = (
    mlflow.sklearn.load_model(
        NEW_VERSION_MODEL_URI
    )
)

registered_version_predictions = (
    registered_challenger_model.predict(
        verification_sample
    )
)

assert np.array_equal(
    pre_registration_predictions,
    registered_version_predictions
), (
    "Predictions changed after Challenger registration."
)

print(
    "Registered Challenger predictions match "
    "the logged model predictions."
)

Registered Challenger predictions match the logged model predictions.


In [0]:
pre_promotion_state = {
    "alias": CHAMPION_ALIAS,
    "version": pre_promotion_alias_version,
    "run_id": pre_promotion_alias_run_id,
    "algorithm": old_champion_algorithm,
    "macro_f1": old_champion_macro_f1
}

print("Pre-promotion state:")
print(pre_promotion_state)

Pre-promotion state:
{'alias': 'champion', 'version': '4', 'run_id': '3fa9c04ecf3d4f1597aa3bd68f650d2f', 'algorithm': 'DecisionTreeClassifier', 'macro_f1': 0.3407115240453506}


In [0]:
assert promotion_eligible

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CHAMPION_ALIAS,
    version=new_champion_version
)

print(
    f"Alias '{CHAMPION_ALIAS}' assigned to "
    f"version {new_champion_version}."
)

Alias 'champion' assigned to version 5.


In [0]:
post_promotion_alias_model = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS
    )
)

post_promotion_alias_version = str(
    post_promotion_alias_model.version
)

post_promotion_alias_run_id = str(
    post_promotion_alias_model.run_id
)

post_promotion_aliases = list(
    post_promotion_alias_model.aliases
)

print("Alias after promotion:", CHAMPION_ALIAS)
print("Version after promotion:", post_promotion_alias_version)
print("Run ID after promotion:", post_promotion_alias_run_id)
print("Aliases on version:", post_promotion_aliases)

Alias after promotion: champion
Version after promotion: 5
Run ID after promotion: 80ac2ed817e940c9a66f4e99f15b08f4
Aliases on version: ['champion']


In [0]:
assert (
    post_promotion_alias_version
    == new_champion_version
), (
    "The champion alias does not point to the "
    "new Logistic Regression version."
)

assert (
    post_promotion_alias_run_id
    == challenger_run_id
), (
    "The promoted alias run ID does not match "
    "the Challenger run ID."
)

assert (
    CHAMPION_ALIAS
    in post_promotion_aliases
)

print(
    "The champion alias now points to the "
    "Logistic Regression model."
)

The champion alias now points to the Logistic Regression model.


In [0]:
PROMOTED_CHAMPION_MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"@{CHAMPION_ALIAS}"
)

print(
    "Promoted Champion alias URI:",
    PROMOTED_CHAMPION_MODEL_URI
)

Promoted Champion alias URI: models:/workspace.default.football_match_result_model@champion


In [0]:
promoted_champion_model = (
    mlflow.sklearn.load_model(
        PROMOTED_CHAMPION_MODEL_URI
    )
)

print(
    "Promoted Champion loaded through alias."
)

if hasattr(
    promoted_champion_model,
    "named_steps"
):

    promoted_classifier_name = type(
        promoted_champion_model.named_steps[
            "classifier"
        ]
    ).__name__

    print(
        "Promoted classifier:",
        promoted_classifier_name
    )

    assert (
        promoted_classifier_name
        == "LogisticRegression"
    )

Promoted Champion loaded through alias.
Promoted classifier: LogisticRegression


In [0]:
promoted_champion_model = (
    mlflow.sklearn.load_model(
        PROMOTED_CHAMPION_MODEL_URI
    )
)

print(
    "Promoted Champion loaded through alias."
)

if hasattr(
    promoted_champion_model,
    "named_steps"
):

    promoted_classifier_name = type(
        promoted_champion_model.named_steps[
            "classifier"
        ]
    ).__name__

    print(
        "Promoted classifier:",
        promoted_classifier_name
    )

    assert (
        promoted_classifier_name
        == "LogisticRegression"
    )

Promoted Champion loaded through alias.
Promoted classifier: LogisticRegression


In [0]:
preserved_old_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=old_champion_version
    )
)

print(
    "Previous Champion version:",
    preserved_old_version.version
)

print(
    "Previous Champion status:",
    preserved_old_version.status
)

print(
    "Previous Champion run ID:",
    preserved_old_version.run_id
)

assert str(
    preserved_old_version.run_id
) == old_champion_run_id

print(
    "The previous Decision Tree version remains "
    "available for lineage and rollback."
)

Previous Champion version: 4
Previous Champion status: READY
Previous Champion run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
The previous Decision Tree version remains available for lineage and rollback.


In [0]:
promotion_record_df = pd.DataFrame(
    [
        {
            "promotion_timestamp_utc": (
                promotion_timestamp_utc
            ),
            "registered_model_name": (
                REGISTERED_MODEL_NAME
            ),
            "promotion_metric": (
                PROMOTION_METRIC
            ),
            "minimum_required_improvement": (
                MINIMUM_REQUIRED_IMPROVEMENT
            ),
            "old_champion_algorithm": (
                old_champion_algorithm
            ),
            "old_champion_version": (
                old_champion_version
            ),
            "old_champion_run_id": (
                old_champion_run_id
            ),
            "old_champion_macro_f1": (
                old_champion_macro_f1
            ),
            "challenger_algorithm": (
                challenger_algorithm
            ),
            "challenger_run_id": (
                challenger_run_id
            ),
            "challenger_source_model_uri": (
                challenger_model_uri
            ),
            "challenger_macro_f1": (
                challenger_macro_f1
            ),
            "absolute_improvement": (
                recalculated_improvement
            ),
            "relative_improvement_percentage": (
                relative_improvement_percentage
            ),
            "promotion_eligible": (
                promotion_eligible
            ),
            "promotion_decision": (
                "PROMOTED"
            ),
            "new_champion_version": (
                new_champion_version
            ),
            "new_champion_alias": (
                CHAMPION_ALIAS
            ),
            "new_champion_alias_uri": (
                PROMOTED_CHAMPION_MODEL_URI
            ),
            "alias_verification": (
                "PASSED"
            ),
            "prediction_verification": (
                "PASSED"
            ),
            "old_version_preserved": True
        }
    ]
)

display(promotion_record_df)

promotion_timestamp_utc,registered_model_name,promotion_metric,minimum_required_improvement,old_champion_algorithm,old_champion_version,old_champion_run_id,old_champion_macro_f1,challenger_algorithm,challenger_run_id,challenger_source_model_uri,challenger_macro_f1,absolute_improvement,relative_improvement_percentage,promotion_eligible,promotion_decision,new_champion_version,new_champion_alias,new_champion_alias_uri,alias_verification,prediction_verification,old_version_preserved
2026-07-27T15:13:08.147727+00:00,workspace.default.football_match_result_model,test_f1_macro,0.0,DecisionTreeClassifier,4,3fa9c04ecf3d4f1597aa3bd68f650d2f,0.3407115240453506,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,0.6222097633743984,0.2814982393290478,82.62069799892615,true,PROMOTED,5,champion,models:/workspace.default.football_match_result_model@champion,PASSED,PASSED,true


In [0]:
promotion_record_df = pd.DataFrame(
    [
        {
            "promotion_timestamp_utc": (
                promotion_timestamp_utc
            ),
            "registered_model_name": (
                REGISTERED_MODEL_NAME
            ),
            "promotion_metric": (
                PROMOTION_METRIC
            ),
            "minimum_required_improvement": (
                MINIMUM_REQUIRED_IMPROVEMENT
            ),
            "old_champion_algorithm": (
                old_champion_algorithm
            ),
            "old_champion_version": (
                old_champion_version
            ),
            "old_champion_run_id": (
                old_champion_run_id
            ),
            "old_champion_macro_f1": (
                old_champion_macro_f1
            ),
            "challenger_algorithm": (
                challenger_algorithm
            ),
            "challenger_run_id": (
                challenger_run_id
            ),
            "challenger_source_model_uri": (
                challenger_model_uri
            ),
            "challenger_macro_f1": (
                challenger_macro_f1
            ),
            "absolute_improvement": (
                recalculated_improvement
            ),
            "relative_improvement_percentage": (
                relative_improvement_percentage
            ),
            "promotion_eligible": (
                promotion_eligible
            ),
            "promotion_decision": (
                "PROMOTED"
            ),
            "new_champion_version": (
                new_champion_version
            ),
            "new_champion_alias": (
                CHAMPION_ALIAS
            ),
            "new_champion_alias_uri": (
                PROMOTED_CHAMPION_MODEL_URI
            ),
            "alias_verification": (
                "PASSED"
            ),
            "prediction_verification": (
                "PASSED"
            ),
            "old_version_preserved": True
        }
    ]
)

display(promotion_record_df)

promotion_timestamp_utc,registered_model_name,promotion_metric,minimum_required_improvement,old_champion_algorithm,old_champion_version,old_champion_run_id,old_champion_macro_f1,challenger_algorithm,challenger_run_id,challenger_source_model_uri,challenger_macro_f1,absolute_improvement,relative_improvement_percentage,promotion_eligible,promotion_decision,new_champion_version,new_champion_alias,new_champion_alias_uri,alias_verification,prediction_verification,old_version_preserved
2026-07-27T15:13:08.147727+00:00,workspace.default.football_match_result_model,test_f1_macro,0.0,DecisionTreeClassifier,4,3fa9c04ecf3d4f1597aa3bd68f650d2f,0.3407115240453506,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,0.6222097633743984,0.2814982393290478,82.62069799892615,true,PROMOTED,5,champion,models:/workspace.default.football_match_result_model@champion,PASSED,PASSED,true


In [0]:
promotion_record_df.to_csv(
    PROMOTION_RECORD_PATH,
    index=False
)

print(
    "Promotion record saved to:",
    PROMOTION_RECORD_PATH
)

Promotion record saved to: /Volumes/workspace/default/football_data/promotion_record.csv


In [0]:
challenger_info_df.loc[
    0,
    "registration_status"
] = "READY"

challenger_info_df.loc[
    0,
    "registered_model_name"
] = REGISTERED_MODEL_NAME

challenger_info_df.loc[
    0,
    "registered_model_version"
] = new_champion_version

challenger_info_df.loc[
    0,
    "registered_model_alias"
] = CHAMPION_ALIAS

challenger_info_df.loc[
    0,
    "alias_model_uri"
] = PROMOTED_CHAMPION_MODEL_URI

challenger_info_df.loc[
    0,
    "alias_status"
] = "assigned"

challenger_info_df.loc[
    0,
    "promotion_status"
] = "promoted"

challenger_info_df.loc[
    0,
    "promotion_timestamp_utc"
] = promotion_timestamp_utc

challenger_info_df.to_csv(
    CHALLENGER_INFO_PATH,
    index=False
)

display(challenger_info_df)

print("Challenger metadata updated.")

model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,prediction_time_seconds,champion_algorithm,champion_run_id,champion_model_version,champion_macro_f1,macro_f1_improvement,relative_improvement_percentage,promotion_eligible,promotion_recommendation,registration_status,registered_model_version,alias_status,registered_model_name,registered_model_alias,alias_model_uri,promotion_status,promotion_timestamp_utc
challenger,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,test_f1_macro,0.6222097633743984,0.6515,0.6446662708271728,0.6136885178601231,0.6222097633743984,0.6434775072688667,15999,4000,25.322230577468872,0.0281131267547607,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506,0.2814982393290478,82.62069799892615,true,PROMOTE_CHALLENGER,READY,5,assigned,workspace.default.football_match_result_model,champion,models:/workspace.default.football_match_result_model@champion,promoted,2026-07-27T15:13:08.147727+00:00


Challenger metadata updated.


In [0]:
new_champion_info_df = challenger_info_df.copy()

new_champion_info_df.loc[
    0,
    "model_role"
] = "champion"

new_champion_info_df.loc[
    0,
    "promotion_recommendation"
] = "PROMOTED"

new_champion_info_df.loc[
    0,
    "previous_champion_algorithm"
] = old_champion_algorithm

new_champion_info_df.loc[
    0,
    "previous_champion_run_id"
] = old_champion_run_id

new_champion_info_df.loc[
    0,
    "previous_champion_version"
] = old_champion_version

new_champion_info_df.loc[
    0,
    "previous_champion_macro_f1"
] = old_champion_macro_f1

new_champion_info_df.loc[
    0,
    "model_uri"
] = challenger_model_uri

new_champion_info_df.loc[
    0,
    "primary_metric"
] = PROMOTION_METRIC

new_champion_info_df.loc[
    0,
    "primary_metric_value"
] = challenger_macro_f1

new_champion_info_df.to_csv(
    CHAMPION_INFO_PATH,
    index=False
)

new_champion_info_df.to_csv(
    NEW_CHAMPION_INFO_PATH,
    index=False
)

display(new_champion_info_df)

print("Champion metadata replaced successfully.")

model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,prediction_time_seconds,champion_algorithm,champion_run_id,champion_model_version,champion_macro_f1,macro_f1_improvement,relative_improvement_percentage,promotion_eligible,promotion_recommendation,registration_status,registered_model_version,alias_status,registered_model_name,registered_model_alias,alias_model_uri,promotion_status,promotion_timestamp_utc,previous_champion_algorithm,previous_champion_run_id,previous_champion_version,previous_champion_macro_f1
champion,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,test_f1_macro,0.6222097633743984,0.6515,0.6446662708271728,0.6136885178601231,0.6222097633743984,0.6434775072688667,15999,4000,25.322230577468872,0.0281131267547607,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506,0.2814982393290478,82.62069799892615,true,PROMOTED,READY,5,assigned,workspace.default.football_match_result_model,champion,models:/workspace.default.football_match_result_model@champion,promoted,2026-07-27T15:13:08.147727+00:00,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506


Champion metadata replaced successfully.


In [0]:
new_registration_info_df = pd.DataFrame(
    [
        {
            "registered_model_name": (
                REGISTERED_MODEL_NAME
            ),
            "registered_model_version": (
                new_champion_version
            ),
            "alias": CHAMPION_ALIAS,
            "alias_model_uri": (
                PROMOTED_CHAMPION_MODEL_URI
            ),
            "algorithm": challenger_algorithm,
            "run_id": challenger_run_id,
            "source_model_uri": (
                challenger_model_uri
            ),
            "primary_metric": (
                PROMOTION_METRIC
            ),
            "primary_metric_value": (
                challenger_macro_f1
            ),
            "registration_status": "READY",
            "alias_status": "assigned",
            "model_role": "champion",
            "previous_champion_version": (
                old_champion_version
            ),
            "previous_champion_run_id": (
                old_champion_run_id
            ),
            "promotion_timestamp_utc": (
                promotion_timestamp_utc
            )
        }
    ]
)

new_registration_info_df.to_csv(
    CHAMPION_REGISTRATION_PATH,
    index=False
)

display(new_registration_info_df)

print(
    "Champion registration information updated."
)

registered_model_name,registered_model_version,alias,alias_model_uri,algorithm,run_id,source_model_uri,primary_metric,primary_metric_value,registration_status,alias_status,model_role,previous_champion_version,previous_champion_run_id,promotion_timestamp_utc
workspace.default.football_match_result_model,5,champion,models:/workspace.default.football_match_result_model@champion,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,models:/m-d7e371663ec74ae8930885ebaa2a6cc2,test_f1_macro,0.6222097633743984,READY,assigned,champion,4,3fa9c04ecf3d4f1597aa3bd68f650d2f,2026-07-27T15:13:08.147727+00:00


Champion registration information updated.


In [0]:
current_history_rows = pd.DataFrame(
    [
        {
            "model_role": "previous_champion",
            "algorithm": old_champion_algorithm,
            "run_id": old_champion_run_id,
            "registered_model_version": (
                old_champion_version
            ),
            "macro_f1": old_champion_macro_f1,
            "alias_status": "replaced",
            "promotion_status": (
                "retained_for_rollback"
            ),
            "record_timestamp_utc": (
                promotion_timestamp_utc
            )
        },
        {
            "model_role": "champion",
            "algorithm": challenger_algorithm,
            "run_id": challenger_run_id,
            "registered_model_version": (
                new_champion_version
            ),
            "macro_f1": challenger_macro_f1,
            "alias_status": "champion",
            "promotion_status": "promoted",
            "record_timestamp_utc": (
                promotion_timestamp_utc
            )
        }
    ]
)

In [0]:
if os.path.exists(MODEL_HISTORY_PATH):

    existing_history_df = pd.read_csv(
        MODEL_HISTORY_PATH
    )

    model_history_df = pd.concat(
        [
            existing_history_df,
            current_history_rows
        ],
        ignore_index=True
    )

    model_history_df = (
        model_history_df.drop_duplicates(
            subset=[
                "run_id",
                "registered_model_version",
                "model_role"
            ],
            keep="last"
        )
    )

else:
    model_history_df = (
        current_history_rows.copy()
    )

# Ensure a consistent Arrow-compatible type
model_history_df["registered_model_version"] = (
    model_history_df["registered_model_version"]
    .astype("string")
)

model_history_df.to_csv(
    MODEL_HISTORY_PATH,
    index=False
)

display(model_history_df)

print(
    "Model history saved to:",
    MODEL_HISTORY_PATH
)

model_role,algorithm,run_id,registered_model_version,macro_f1,alias_status,promotion_status,record_timestamp_utc
previous_champion,DecisionTreeClassifier,36362b3ba7c6483f8f3f3e9d22c555da,2,0.3407115240453506,replaced,retained_for_rollback,2026-07-26T00:54:09.050620+00:00
champion,LogisticRegression,274498f94f3145bdbfcc664a5dd9a952,3,0.6222097633743984,champion,promoted,2026-07-26T00:54:09.050620+00:00
previous_champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506,replaced,retained_for_rollback,2026-07-27T15:13:08.147727+00:00
champion,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,5,0.6222097633743984,champion,promoted,2026-07-27T15:13:08.147727+00:00
previous_champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,4,0.3407115240453506,replaced,retained_for_rollback,2026-07-27T15:13:08.147727+00:00
champion,LogisticRegression,80ac2ed817e940c9a66f4e99f15b08f4,5,0.6222097633743984,champion,promoted,2026-07-27T15:13:08.147727+00:00


Model history saved to: /Volumes/workspace/default/football_data/model_history.csv


In [0]:
required_output_files = [
    PROMOTION_RECORD_PATH,
    MODEL_HISTORY_PATH,
    NEW_CHAMPION_INFO_PATH,
    CHAMPION_INFO_PATH,
    CHAMPION_REGISTRATION_PATH,
    CHALLENGER_INFO_PATH
]

missing_output_files = [
    file_path
    for file_path in required_output_files
    if not os.path.exists(file_path)
]

assert not missing_output_files, (
    "The following output files are missing: "
    f"{missing_output_files}"
)

saved_promotion_record = pd.read_csv(
    PROMOTION_RECORD_PATH
)

saved_champion_info = pd.read_csv(
    CHAMPION_INFO_PATH
)

saved_registration_info = pd.read_csv(
    CHAMPION_REGISTRATION_PATH
)

assert len(saved_promotion_record) == 1
assert len(saved_champion_info) == 1
assert len(saved_registration_info) == 1

assert (
    saved_champion_info.iloc[0][
        "algorithm"
    ]
    == "LogisticRegression"
)

assert str(
    saved_champion_info.iloc[0][
        "registered_model_version"
    ]
) == new_champion_version

assert (
    saved_champion_info.iloc[0][
        "registered_model_alias"
    ]
    == CHAMPION_ALIAS
)

assert (
    saved_promotion_record.iloc[0][
        "promotion_decision"
    ]
    == "PROMOTED"
)

print("Updated project files validated.")

Updated project files validated.


In [0]:
final_alias_model = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS
    )
)

final_alias_version = str(
    final_alias_model.version
)

final_alias_run_id = str(
    final_alias_model.run_id
)

assert final_alias_version == new_champion_version

assert final_alias_run_id == challenger_run_id

assert final_alias_version != old_champion_version

assert final_alias_run_id != old_champion_run_id

print("Final registry verification passed.")

Final registry verification passed.


In [0]:
print("=" * 72)
print("NOTEBOOK 7 COMPLETED SUCCESSFULLY")
print("=" * 72)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME
)

print("\nBEFORE PROMOTION")

print(
    "Champion algorithm:",
    old_champion_algorithm
)

print(
    "Champion version:",
    old_champion_version
)

print(
    "Champion run ID:",
    old_champion_run_id
)

print(
    "Champion Macro F1:",
    round(old_champion_macro_f1, 4)
)

print("\nAFTER PROMOTION")

print(
    "New Champion algorithm:",
    challenger_algorithm
)

print(
    "New Champion version:",
    new_champion_version
)

print(
    "New Champion run ID:",
    challenger_run_id
)

print(
    "New Champion Macro F1:",
    round(challenger_macro_f1, 4)
)

print(
    "Absolute improvement:",
    round(recalculated_improvement, 4)
)

print(
    "Relative improvement:",
    round(
        relative_improvement_percentage,
        2
    ),
    "%"
)

print(
    "Champion alias:",
    CHAMPION_ALIAS
)

print(
    "Champion alias URI:",
    PROMOTED_CHAMPION_MODEL_URI
)

print(
    "Alias now resolves to version:",
    final_alias_version
)

print(
    "Previous Champion preserved:",
    True
)

print(
    "Alias verification:",
    "PASSED"
)

print(
    "Prediction verification:",
    "PASSED"
)

print(
    "Promotion record:",
    PROMOTION_RECORD_PATH
)

print(
    "Model history:",
    MODEL_HISTORY_PATH
)

print(
    "Result: Logistic Regression has been "
    "automatically promoted as the new Champion."
)

NOTEBOOK 7 COMPLETED SUCCESSFULLY
Registered model: workspace.default.football_match_result_model

BEFORE PROMOTION
Champion algorithm: DecisionTreeClassifier
Champion version: 4
Champion run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
Champion Macro F1: 0.3407

AFTER PROMOTION
New Champion algorithm: LogisticRegression
New Champion version: 5
New Champion run ID: 80ac2ed817e940c9a66f4e99f15b08f4
New Champion Macro F1: 0.6222
Absolute improvement: 0.2815
Relative improvement: 82.62 %
Champion alias: champion
Champion alias URI: models:/workspace.default.football_match_result_model@champion
Alias now resolves to version: 5
Previous Champion preserved: True
Alias verification: PASSED
Prediction verification: PASSED
Promotion record: /Volumes/workspace/default/football_data/promotion_record.csv
Model history: /Volumes/workspace/default/football_data/model_history.csv
Result: Logistic Regression has been automatically promoted as the new Champion.
